# Mirror Param Calculation
## Consts:

In [15]:
cameraAlpha: int = 20 # half angle of camera - deg
mirrorHeightFromGround: int = 180 # mm
fieldRadius: int = 2500 # mm

hyperbolicC: float = 25 # mm

amountOfPoints: int = 30

## Calculations:

In [9]:
import math

beta = math.degrees(math.atan(fieldRadius / mirrorHeightFromGround))

class Calculations:
    @staticmethod
    def cot(deg: float) -> float:
        return 1 / math.tan(math.radians(deg))

    @staticmethod
    def mirrorRadius(c: float) -> float:
        return 2 * c / (Calculations.cot(cameraAlpha) + Calculations.cot(beta))

    @staticmethod
    def mirrorZmax(c: float) -> float:
        return c - Calculations.mirrorRadius(c) * Calculations.cot(beta)

    @staticmethod
    def hyperbolicA(c: float) -> float:
        quadraticEquationB = (c ** 2 + Calculations.mirrorZmax(c) ** 2 + Calculations.mirrorRadius(c) ** 2)
        quadraticEquationC = (Calculations.mirrorZmax(c) ** 2) * (c ** 2)
        return math.sqrt((quadraticEquationB - math.sqrt(quadraticEquationB ** 2 - 4 * quadraticEquationC)) / 2)

    @staticmethod
    def hyperbolicB(c: float) -> float:
        return math.sqrt(c ** 2 - Calculations.hyperbolicA(c) ** 2)

    @staticmethod
    def mirrorHeight(c: float) -> float:
        return Calculations.mirrorZmax(c) * Calculations.hyperbolicA(c)


## Run

In [10]:
print(Calculations.hyperbolicA(hyperbolicC), Calculations.hyperbolicB(hyperbolicC))

17.035211360973516 18.29758382647717


# Hyperbolic Function Drawing in DXF
## Hyperbolic Function:

In [11]:
def hyperbolicFunction(x: float, a: float, b: float) -> float: #returns y
    return (a * math.sqrt(1 + (x ** 2) / (b ** 2))) - a

## Points Calculations

In [17]:
radius = Calculations.mirrorRadius(hyperbolicC)
hyperbolicA = Calculations.hyperbolicA(hyperbolicC)
hyperbolicB = Calculations.hyperbolicB(hyperbolicC)

pointsX = [radius * ((i / amountOfPoints) ** 1.5) for i in range(0, amountOfPoints)]
pointsY = [hyperbolicFunction(point, hyperbolicA, hyperbolicB) for point in pointsX]

points = [(pointX, pointY) for pointX, pointY in zip(pointsX, pointsY)]


## DXF Export

In [ ]:
!pip install ezdxf

In [22]:
import ezdxf
from ezdxf.units import MM

doc = ezdxf.new(setup=True)
doc.units = MM
msp = doc.modelspace()

layerName: str = "MIRROR_PROFILE"

if layerName not in doc.layers:
    doc.layers.add(layerName)

points3D = [(x, y, 0.0) for x, y in points]

msp.add_spline(fit_points=points3D, dxfattribs={"layer": layerName})

doc.saveas("mirror.dxf")
print("Finished exporting mirror profile")

Finished exporting mirror profile
